In [1]:
#import all sorts of packages 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

In [2]:
random_st = 41

In [3]:
data = pd.read_csv("data/final_offshore_data_2017_2025.csv", dtype = {'year_mon_day':'int32'})
# Removing data from 2025
data = data[data['full_datetime'] < "2025-01-01-01"]

# Fix function
def fix_datetime_format(date_str):
    parts = date_str.split("-")
    year, month, day, hour = map(int, parts)  # Convert all parts to integers
    
    if hour == 24:
        hour = 0  # Reset hour to 00
        new_date = pd.Timestamp(year, month, day) + pd.Timedelta(days=1)  # Add a day
        return new_date.replace(hour=hour)  # Set hour to 00
    else:
        return pd.Timestamp(year, month, day, hour)

# Apply the fix
data["correct_days"] = data["correct_days"].apply(fix_datetime_format)
# Create a unique numerical index
#data['unique_id'] = range(1, len(data)+1)  # Ensuring uniqueness

data.tail()

,year_mon_day,hour,wind_dir_avg_10,wind_speed_h_avg,wind_speed_avg_10,air_pressure,humidity,full_datetime,capacity,volume,percentage,emission,emissionfactor,correct_days
70123,20241231,20,211.999178,146.000000,151.333333,10141.5,86.571429,2024-12-31-20,3718000,3718000,0.894180,0,0,2024-12-31 20:00:00
70124,20241231,21,210.666633,146.000000,148.000000,10133.5,86.857143,2024-12-31-21,3744500,3744500,0.900553,0,0,2024-12-31 21:00:00
70125,20241231,22,210.000000,146.000000,149.333333,10124.0,88.285714,2024-12-31-22,3686750,3686750,0.886664,0,0,2024-12-31 22:00:00
70126,20241231,23,213.336340,150.000000,149.333333,10118.5,88.857143,2024-12-31-23,3719250,3719250,0.894481,0,0,2024-12-31 23:00:00
70127,20241231,24,214.005693,152.666667,153.333333,10109.5,87.428571,2024-12-31-24,3453249,3453249,0.830507,0,0,2025-01-01 00:00:00


In [19]:

data.info()


<class 'pandas.core.frame.DataFrame'>
Int64Index: 70128 entries, 0 to 70127
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   year_mon_day       70128 non-null  int32         
 1   hour               70128 non-null  int64         
 2   wind_dir_avg_10    70128 non-null  float64       
 3   wind_speed_h_avg   70128 non-null  float64       
 4   wind_speed_avg_10  70128 non-null  float64       
 5   air_pressure       70128 non-null  float64       
 6   humidity           70128 non-null  float64       
 7   full_datetime      70128 non-null  object        
 8   capacity           70128 non-null  int64         
 9   volume             70128 non-null  int64         
 10  percentage         70128 non-null  float64       
 11  emission           70128 non-null  int64         
 12  emissionfactor     70128 non-null  int64         
 13  correct_days       70128 non-null  datetime64[ns]
dtypes: dat

In [37]:
# Split data into X (features) and y (label)
X = data.drop(["volume", "capacity", "emission", "emissionfactor", "full_datetime", "percentage", "hour","year_mon_day"], 
              #capacity & volume are the same, emission and emissionfactor are all zeros
                      axis=1)


# Set MultiIndex (unique_id as first, correct_days as second)
X.set_index(['correct_days'], inplace=True)
#X.index = pd.MultiIndex.from_frame(X[['year_mon_day', 'correct_days']])


# Sorting by MultiIndex (optional but useful)
#X.sort_index(inplace=True)
#X = X.astype('float16')

y = data.set_index('correct_days').loc[:, 'volume']

In [42]:
X.columns

Index(['wind_dir_avg_10', 'wind_speed_h_avg', 'wind_speed_avg_10',
       'air_pressure', 'humidity'],
      dtype='object')

## Splitting the data

In [39]:
# Split data based on the index
# Split data based on the 'full_datetime' index level
X_train = X.loc[X.index.get_level_values('correct_days') < "2024-01-01 01:00:00"]
X_test = X.loc[X.index.get_level_values('correct_days') >= "2024-01-01 01:00:00"]

y_train = y.loc[y.index < "2024-01-01 01:00:00"]
y_test = y.loc[y.index >= "2024-01-01 01:00:00"]

In [9]:
X_train.tail()

wind_dir_avg_10  wind_speed_h_avg  \
year_mon_day correct_days                                             
20231231     2023-12-31 20:00:00          198.625          130.0000   
             2023-12-31 21:00:00          200.000          122.1250   
             2023-12-31 22:00:00          197.125          115.6875   
             2023-12-31 23:00:00          197.875          119.3125   
             2024-01-01 00:00:00          203.625          120.0000   

                                  wind_speed_avg_10  air_pressure  humidity  
year_mon_day correct_days                                                    
20231231     2023-12-31 20:00:00           128.6250        9920.0   83.3125  
             2023-12-31 21:00:00           120.6875        9928.0   86.4375  
             2023-12-31 22:00:00           117.1250        9928.0   88.4375  
             2023-12-31 23:00:00           117.1250        9928.0   88.6875  
             2024-01-01 00:00:00           118.5625        9928.0   87.8750

## Preprocessing

Although normalization is not required for tree-based methods, it can enhance numerical stability, so let us implement it.

In [40]:
# Define the preprocessor with feature names retained
preprocessor = StandardScaler()

# Fit and transform the data
X_train_norm = preprocessor.fit_transform(X_train)
X_test_norm = preprocessor.transform(X_test)

# Convert back to DataFrame with original column names
feature_names = preprocessor.get_feature_names_out()
X_train_norm = pd.DataFrame(X_train_norm, columns=X_train.columns, index=X_train.index)
X_test_norm = pd.DataFrame(X_test_norm, columns=X_test.columns, index=X_test.index)

In [47]:
# Assuming window size of 24 hours for feature extraction
window_size = 24
n_samples = len(X_train_norm) - window_size + 1

# Creating nested DataFrame for sktime
X_nested_train = pd.DataFrame(index=np.arange(n_samples), columns=['wind_dir_avg_10', 'wind_speed_h_avg', 'wind_speed_avg_10',
       'air_pressure', 'humidity'])
for i in range(n_samples):
    X_nested_train.iloc[i, 0] = pd.Series(X_train_norm['wind_dir_avg_10'].iloc[i:i+window_size].values)
    X_nested_train.iloc[i, 1] = pd.Series(X_train_norm['wind_speed_h_avg'].iloc[i:i+window_size].values)
    X_nested_train.iloc[i, 2] = pd.Series(X_train_norm['wind_speed_avg_10'].iloc[i:i+window_size].values)
    X_nested_train.iloc[i, 3] = pd.Series(X_train_norm['air_pressure'].iloc[i:i+window_size].values)
    X_nested_train.iloc[i, 4] = pd.Series(X_train_norm['humidity'].iloc[i:i+window_size].values)

n_samples = len(X_test_norm) - window_size + 1
X_nested_test = pd.DataFrame(index=np.arange(n_samples), columns=['wind_dir_avg_10', 'wind_speed_h_avg', 'wind_speed_avg_10',
       'air_pressure', 'humidity'])
for i in range(n_samples):
    X_nested_test.iloc[i, 0] = pd.Series(X_train_norm['wind_dir_avg_10'].iloc[i:i+window_size].values)
    X_nested_test.iloc[i, 1] = pd.Series(X_train_norm['wind_speed_h_avg'].iloc[i:i+window_size].values)
    X_nested_test.iloc[i, 2] = pd.Series(X_train_norm['wind_speed_avg_10'].iloc[i:i+window_size].values)
    X_nested_test.iloc[i, 3] = pd.Series(X_train_norm['air_pressure'].iloc[i:i+window_size].values)
    X_nested_test.iloc[i, 4] = pd.Series(X_train_norm['humidity'].iloc[i:i+window_size].values)


In [66]:
X_nested_train

,wind_dir_avg_10,wind_speed_h_avg,wind_speed_avg_10,air_pressure,humidity
0,"[0.23713527010854496, 0.20935342831932058, 0.1...","[0.6755876009509005, 0.7252350421328795, 0.799...","[0.6960004414259879, 0.8446426920043544, 0.745...","[0.5387444176867756, 0.47421158564095567, 0.39...","[1.2961468321054868, 1.334246132025357, 1.3215..."
1,"[0.20935342831932058, 0.18531405507610513, 0.1...","[0.7252350421328795, 0.7997062039058481, 0.824...","[0.8446426920043544, 0.7455478582854435, 0.819...","[0.47421158564095567, 0.3981550335869536, 0.31...","[1.334246132025357, 1.3215463653854, 1.3342461..."
2,"[0.18531405507610513, 0.19974597968261326, 0.1...","[0.7997062039058481, 0.8245299244968374, 0.799...","[0.7455478582854435, 0.8198689835746265, 0.720...","[0.3981550335869536, 0.3128795055264058, 0.257...","[1.3215463653854, 1.334246132025357, 1.2834470..."
3,"[0.19974597968261326, 0.18486407373429423, 0.2...","[0.8245299244968374, 0.7997062039058481, 0.899...","[0.8198689835746265, 0.7207741498557154, 0.943...","[0.3128795055264058, 0.25756564948713156, 0.19...","[1.334246132025357, 1.2834470654655274, 1.2580..."
4,"[0.18486407373429423, 0.2074250211202233, 0.22...","[0.7997062039058481, 0.8990010862698056, 0.973...","[0.7207741498557154, 0.9437375257232644, 0.968...","[0.25756564948713156, 0.19533756144294803, 0.1...","[1.2834470654655274, 1.2580475321856142, 1.270..."
...,...,...,...,...,...
61316,"[-0.16996473175818688, -0.14796907523753197, -...","[0.8245299244968374, 1.047943409815743, 1.1472...","[0.9189638172935375, 1.0676060678719035, 1.265...","[-1.0031293194079938, -1.0791858714619957, -1....","[0.5595603669879774, 0.5849599002678908, 0.559..."
61317,"[-0.14796907523753197, -0.16996473175818688, -...","[1.047943409815743, 1.1472382921796997, 1.2217...","[1.0676060678719035, 1.2657957353097256, 1.166...","[-1.0791858714619957, -1.1460234475094522, -1....","[0.5849599002678908, 0.5595603669879774, 0.572..."
61318,"[-0.16996473175818688, -0.15532074445009564, -...","[1.1472382921796997, 1.221709453952669, 1.2465...","[1.2657957353097256, 1.1667009015908145, 1.191...","[-1.1460234475094522, -1.2520416715847278, -1....","[0.5595603669879774, 0.5722601336279342, 0.483..."
61319,"[-0.15532074445009564, -0.19224650927804005, -...","[1.221709453952669, 1.2465331745436576, 1.2465...","[1.1667009015908145, 1.1914746100205424, 1.191...","[-1.2520416715847278, -1.3465361756518213, -1....","[0.5722601336279342, 0.48336176714823476, 0.38..."


## Random forest model

In [210]:
rf = RandomForestRegressor(random_state = random_st,
                           max_depth = 10, n_estimators = 500, min_samples_leaf = 5)
rf.fit(X_train_norm, y_train)
y_pred_rf = rf.predict(X_test_norm)

print(mean_squared_error(y_test, y_pred_rf))
print(mean_squared_error(y_test, y_pred_rf)**0.5)

ValueError: Input X contains NaN.
RandomForestRegressor does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
rf

In [ ]:
# Grid search for hyperparameter tuning
param_grid = {
    #'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    #'min_samples_split': [2, 5, 10]
}

rf = RandomForestRegressor(random_state=42)
cv_rf = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5) #Here we perform a grid search with 5-fold cross validation
cv_rf.fit(X_train, y_train) # Fit the model to the training set - this will perform the grid search and find the model with the best hyper-parameters

# Mean cross-validated score of the best_estimator
print("Mean cross-validated score of the best_estimator:", cv_rf.best_score_)

# Print the best parameters
print("Best parameters:", cv_rf.best_params_)

# Refit the model with the best parameters on the entire training set
rf_model = cv_rf.best_estimator_
rf_pred = rf_model.predict(X_test)

#Calculating the accuracy
print("Best Random Forest Classifier")
print("Test Set MSE: ", mean_squared_error(y_test, rf_pred))

In [77]:
my_df = data.drop(["capacity", "emission", "emissionfactor", "year_mon_day", "percentage", "hour", "correct_days"], 
              #capacity & volume are the same, emission and emissionfactor are all zeros
                      axis=1)

my_df.to_csv("offshore_rf.csv", index = False)

In [86]:
# forecast monthly births with random forest
from numpy import asarray
from pandas import read_csv
from pandas import DataFrame
from pandas import concat
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from matplotlib import pyplot

# transform a time series dataset into a supervised learning dataset
def series_to_supervised(data, n_in=24, n_out=1, dropnan=True):
    n_vars = 1 if type(data) is list else data.shape[1]
    df = DataFrame(data)
    cols = list()
    # input sequence (t-n, ... t-1)
    for i in range(n_in, 0, -1):
        cols.append(df.shift(i))
    # forecast sequence (t, t+1, ... t+n)
    for i in range(0, n_out):
        cols.append(df.shift(-i))
    # put it all together
    agg = concat(cols, axis=1)
    # drop rows with NaN values
    if dropnan:
        agg.dropna(inplace=True)
    return agg.values

# split a univariate dataset into train/test sets
def train_test_split(data, n_test):
    return data[:-n_test, :], data[-n_test:, :]

# fit an random forest model and make a one step prediction
def random_forest_forecast(train, testX):
    # transform list into array
    train = asarray(train)
    # split into input and output columns
    trainX, trainy = train[:, :-1], train[:, -1]
    # fit model
    model = RandomForestRegressor(n_estimators=500, max_depth = 10, min_samples_leaf = 5, random_state = random_st)
    model.fit(trainX, trainy)
    # make a one-step prediction
    yhat = model.predict([testX])
    return yhat[0]

# walk-forward validation for univariate data
def walk_forward_validation(data, n_test):
    predictions = list()
    # split dataset
    train, test = train_test_split(data, n_test)
    # seed history with training dataset
    history = [x for x in train]
    # step over each time-step in the test set
    for i in range(len(test)):
        # split test row into input and output columns
        testX, testy = test[i, :-1], test[i, -1]
        # fit model on history and make a prediction
        yhat = random_forest_forecast(history, testX)
        # store forecast in list of predictions
        predictions.append(yhat)
        # add actual observation to history for the next loop
        history.append(test[i])
    # estimate prediction error
    error_mae = mean_absolute_error(test[:, -1], predictions)
    error_mse = mean_squared_error(test[:, -1], predictions)
    error_rmse = error_mse**0.5
    return error_mae, error_mse, error_rmse, test[:, -1], predictions


In [87]:
# load the dataset
series = read_csv('offshore_rf.csv', header=0, index_col=0 )
values = series.values
# transform the time series data into supervised learning
data = series_to_supervised(values, n_in=6)

In [88]:
# evaluate
mae, mse, rmse, y, yhat = walk_forward_validation(data, 8784)
print('MAE: %.3f' % mae)
print('MSE: %.3f' % mse)
print('RMSE: %.3f' % rmse)
# plot expected vs predicted
pyplot.plot(y, label='Expected')
pyplot.plot(yhat, label='Predicted')
pyplot.legend()
pyplot.show()

KeyboardInterrupt: 

## sktime attempts

In [8]:
#!pip install sktime

Defaulting to user installation because normal site-packages is not writeable
                                              0.0/36.9 MB ? eta -:--:--
                                              0.1/36.9 MB 2.6 MB/s eta 0:00:14
                                              0.2/36.9 MB 3.0 MB/s eta 0:00:13
                                              0.4/36.9 MB 2.9 MB/s eta 0:00:13
                                              0.5/36.9 MB 3.0 MB/s eta 0:00:13
                                              0.7/36.9 MB 3.1 MB/s eta 0:00:12
     -                                        1.0/36.9 MB 3.4 MB/s eta 0:00:11
     -                                        1.2/36.9 MB 3.6 MB/s eta 0:00:10
     -                                        1.5/36.9 MB 4.0 MB/s eta 0:00:09
     -                                        1.7/36.9 MB 4.1 MB/s eta 0:00:09
     --                                       1.9/36.9 MB 4.2 MB/s eta 0:00:09
     --                                       2.1/36.9 MB 4.

In [63]:
from sktime.regression.distance_based import KNeighborsTimeSeriesRegressor
from sktime.datasets import load_unit_test

In [49]:
from sktime.datatypes import convert_to
import numpy as np

# Convert MultiIndex DataFrame to 3D NumPy array
X_train_panel = convert_to(X_nested_train, to_type="numpy3D", as_scitype="Panel")
X_test_panel = convert_to(X_nested_test, to_type="numpy3D", as_scitype="Panel")

print("X_train_panel shape:", X_train_panel.shape)  # (n_instances, n_features, n_timepoints)
print("X_test_panel shape:", X_test_panel.shape)




TypeError: No valid mtype could be identified for object of type <class 'pandas.core.frame.DataFrame'>. Errors returned are as follows, in format [mtype]: [error message] 
dask_panel: obj must be of python type dask.dataframe, or a subtype thereof, but found <class 'pandas.core.frame.DataFrame'>
df-list: obj must be of python type list, or a subtype thereof, but found <class 'pandas.core.frame.DataFrame'>
gluonts_ListDataset_panel: obj must be of python type list, or a subtype thereof, but found <class 'pandas.core.frame.DataFrame'>
nested_univ: obj entries must be pd.Series
numpy3D: obj must be of python type numpy.ndarray, or a subtype thereof, but found <class 'pandas.core.frame.DataFrame'>
pd-multiindex: obj must have a MultiIndex, found <class 'pandas.core.indexes.numeric.Int64Index'>


In [83]:
from sktime.datasets import load_unit_test
from sktime.regression.distance_based import KNeighborsTimeSeriesRegressor

X_nested_train_fixed = X_nested_train.values.reshape(X_nested_train.shape[0], X_nested_train.shape[1])

regressor = KNeighborsTimeSeriesRegressor(n_neighbors = 5)
regressor.fit(X_nested_train_fixed, y_train)
#y_pred = regressor.predict(X_nested_test)

ValueError: Mismatch in number of cases. Number in X = 61321 nos in y = 61344

In [85]:
X_nested_train_fixed.shape

(61321, 5)